# PayShield AI — Data Generation

### AI-Powered Payment Incident Prediction & Prevention

This notebook generates a synthetic payment transaction dataset
for developing and testing PayShield AI.

The dataset simulates:
- UPI transactions
- Multiple banks
- Different payment applications
- Payment success and failure
- Bank degradation
- Severe incidents
- Recovery periods
- Transaction latency
- Payment errors

In [12]:
import numpy as np
import pandas as pd

np.random.seed(42)

NUM_TRANSACTIONS = 100_000

banks = [
    "HDFC",
    "ICICI",
    "SBI",
    "AXIS",
    "KOTAK",
    "PNB",
    "BOB",
    "CANARA",
    "IDFC",
    "INDUSIND"
]

upi_apps = [
    "PhonePe",
    "GooglePay",
    "Paytm",
    "BHIM"
]

failure_reasons = [
    "BANK_TIMEOUT",
    "BANK_SERVER_ERROR",
    "NETWORK_TIMEOUT",
    "MERCHANT_ERROR",
    "INSUFFICIENT_FUNDS",
    "INVALID_UPI"
]

In [13]:
start_time = pd.Timestamp("2026-01-01")
end_time = pd.Timestamp("2026-01-30 23:59:59")

total_seconds = int(
    (end_time - start_time).total_seconds()
)

random_seconds = np.random.randint(
    0,
    total_seconds,
    NUM_TRANSACTIONS
)

timestamps = (
    start_time +
    pd.to_timedelta(random_seconds, unit="s")
)

timestamps = pd.Series(timestamps)

In [4]:
df = pd.DataFrame({
    "transaction_id": [
        f"TXN{i:07d}"
        for i in range(1, NUM_TRANSACTIONS + 1)
    ],

    "timestamp": timestamps,

    "amount": np.round(
        np.random.lognormal(
            mean=6,
            sigma=1,
            size=NUM_TRANSACTIONS
        ),
        2
    ),

    "sender_bank": np.random.choice(
        banks,
        NUM_TRANSACTIONS
    ),

    "receiver_bank": np.random.choice(
        banks,
        NUM_TRANSACTIONS
    ),

    "payment_method": np.random.choice(
        payment_methods,
        NUM_TRANSACTIONS
    ),

    "upi_app": np.random.choice(
        upi_apps,
        NUM_TRANSACTIONS
    )
})

df.head()

,transaction_id,timestamp,amount,sender_bank,receiver_bank,payment_method,upi_app
0,TXN0000001,2026-01-01 00:00:37,472.69,AXIS,CANARA,UPI,PhonePe
1,TXN0000002,2026-01-01 00:00:56,1287.24,PNB,PNB,UPI,Paytm
2,TXN0000003,2026-01-01 00:00:59,587.20,HDFC,HDFC,UPI,GooglePay
3,TXN0000004,2026-01-01 00:01:09,145.57,BOB,PNB,UPI,PhonePe
4,TXN0000005,2026-01-01 00:01:13,574.99,INDUSIND,HDFC,UPI,GooglePay


In [5]:
df["bank_health"] = "NORMAL"

incidents = [
    ("SBI", "2026-01-05 10:00", "2026-01-05 18:00"),
    ("HDFC", "2026-01-10 14:00", "2026-01-10 22:00"),
    ("AXIS", "2026-01-15 08:00", "2026-01-15 16:00"),
    ("ICICI", "2026-01-20 10:00", "2026-01-20 18:00"),
    ("KOTAK", "2026-01-25 12:00", "2026-01-25 20:00"),
]

for bank, start, end in incidents:

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    duration = end - start

    degraded_start = start
    severe_start = start + duration * 0.25
    recovery_start = start + duration * 0.75

    mask_degraded = (
        (df["receiver_bank"] == bank) &
        (df["timestamp"] >= degraded_start) &
        (df["timestamp"] < severe_start)
    )

    df.loc[mask_degraded, "bank_health"] = "DEGRADED"

    mask_severe = (
        (df["receiver_bank"] == bank) &
        (df["timestamp"] >= severe_start) &
        (df["timestamp"] < recovery_start)
    )

    df.loc[mask_severe, "bank_health"] = "SEVERE"

    mask_recovery = (
        (df["receiver_bank"] == bank) &
        (df["timestamp"] >= recovery_start) &
        (df["timestamp"] <= end)
    )

    df.loc[mask_recovery, "bank_health"] = "RECOVERY"

df["bank_health"].value_counts()

bank_health
NORMAL      99431
SEVERE        283
DEGRADED      148
RECOVERY      138
Name: count, dtype: int64

In [6]:
df["latency_ms"] = np.random.normal(
    loc=1200,
    scale=300,
    size=NUM_TRANSACTIONS
)

df["latency_ms"] = np.maximum(
    df["latency_ms"],
    100
)

df.loc[
    df["bank_health"] == "DEGRADED",
    "latency_ms"
] *= np.random.uniform(
    2,
    5,
    size=(df["bank_health"] == "DEGRADED").sum()
)

df.loc[
    df["bank_health"] == "SEVERE",
    "latency_ms"
] *= np.random.uniform(
    4,
    8,
    size=(df["bank_health"] == "SEVERE").sum()
)

In [7]:
success_probability = np.full(
    NUM_TRANSACTIONS,
    0.98
)

success_probability[
    df["bank_health"] == "DEGRADED"
] = 0.80

success_probability[
    df["bank_health"] == "SEVERE"
] = 0.50

success_probability[
    df["bank_health"] == "RECOVERY"
] = 0.85

random_values = np.random.random(
    NUM_TRANSACTIONS
)

df["payment_status"] = np.where(
    random_values < success_probability,
    "SUCCESS",
    "FAILED"
)

In [8]:
df["error_code"] = "NONE"

failed_mask = df["payment_status"] == "FAILED"

df.loc[failed_mask, "error_code"] = np.random.choice(
    failure_reasons,
    failed_mask.sum()
)

In [9]:
df["timeout"] = (
    df["latency_ms"] > 3000
).astype(int)

In [10]:
print("Dataset shape:")
print(df.shape)

print("\nPayment status:")
print(df["payment_status"].value_counts())

print("\nBank health:")
print(df["bank_health"].value_counts())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape:
(100000, 12)

Payment status:
payment_status
SUCCESS    97799
FAILED      2201
Name: count, dtype: int64

Bank health:
bank_health
NORMAL      99431
SEVERE        283
DEGRADED      148
RECOVERY      138
Name: count, dtype: int64

First 5 rows:


,transaction_id,timestamp,amount,sender_bank,receiver_bank,payment_method,upi_app,bank_health,latency_ms,payment_status,error_code,timeout
0,TXN0000001,2026-01-01 00:00:37,472.69,AXIS,CANARA,UPI,PhonePe,NORMAL,604.531470,SUCCESS,NONE,0
1,TXN0000002,2026-01-01 00:00:56,1287.24,PNB,PNB,UPI,Paytm,NORMAL,1197.074588,FAILED,BANK_SERVER_ERROR,0
2,TXN0000003,2026-01-01 00:00:59,587.20,HDFC,HDFC,UPI,GooglePay,NORMAL,1271.020534,SUCCESS,NONE,0
3,TXN0000004,2026-01-01 00:01:09,145.57,BOB,PNB,UPI,PhonePe,NORMAL,957.377173,SUCCESS,NONE,0
4,TXN0000005,2026-01-01 00:01:13,574.99,INDUSIND,HDFC,UPI,GooglePay,NORMAL,1165.837966,SUCCESS,NONE,0


In [11]:
df.to_csv(
    "../data/raw/transactions.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [14]:
# -----------------------------
# Generate realistic timestamps
# -----------------------------

start_date = pd.Timestamp("2026-01-01")
days = 30

# Hourly traffic weights
hour_weights = np.array([
    0.3, 0.3, 0.3, 0.3, 0.4, 0.5,
    0.7, 1.0, 1.5, 2.5, 2.5, 2.0,
    2.0, 2.0, 2.0, 2.0, 2.5, 3.5,
    4.0, 4.0, 3.5, 2.5, 1.5, 0.7
])

# Choose day
random_days = np.random.randint(
    0,
    days,
    NUM_TRANSACTIONS
)

# Choose hour according to traffic intensity
random_hours = np.random.choice(
    np.arange(24),
    size=NUM_TRANSACTIONS,
    p=hour_weights / hour_weights.sum()
)

# Random minute and second
random_minutes = np.random.randint(
    0,
    60,
    NUM_TRANSACTIONS
)

random_seconds = np.random.randint(
    0,
    60,
    NUM_TRANSACTIONS
)

timestamps = (
    start_date
    + pd.to_timedelta(random_days, unit="D")
    + pd.to_timedelta(random_hours, unit="h")
    + pd.to_timedelta(random_minutes, unit="m")
    + pd.to_timedelta(random_seconds, unit="s")
)

timestamps = pd.Series(timestamps).sort_values().reset_index(drop=True)

timestamps.head()

0   2026-01-01 00:01:39
1   2026-01-01 00:07:00
2   2026-01-01 00:17:34
3   2026-01-01 00:20:44
4   2026-01-01 00:35:07
dtype: datetime64[ns]

In [15]:
df = pd.DataFrame({
    "transaction_id": [
        f"TXN{i:07d}"
        for i in range(1, NUM_TRANSACTIONS + 1)
    ],

    "timestamp": timestamps,

    "amount": np.round(
        np.random.lognormal(
            mean=6,
            sigma=1,
            size=NUM_TRANSACTIONS
        ),
        2
    ),

    "sender_bank": np.random.choice(
        banks,
        NUM_TRANSACTIONS
    ),

    "receiver_bank": np.random.choice(
        banks,
        NUM_TRANSACTIONS
    ),

    "payment_method": "UPI",

    "upi_app": np.random.choice(
        upi_apps,
        NUM_TRANSACTIONS
    )
})

df.head()

,transaction_id,timestamp,amount,sender_bank,receiver_bank,payment_method,upi_app
0,TXN0000001,2026-01-01 00:01:39,345.00,AXIS,AXIS,UPI,PhonePe
1,TXN0000002,2026-01-01 00:07:00,2940.21,ICICI,KOTAK,UPI,BHIM
2,TXN0000003,2026-01-01 00:17:34,219.04,INDUSIND,PNB,UPI,PhonePe
3,TXN0000004,2026-01-01 00:20:44,232.49,AXIS,INDUSIND,UPI,Paytm
4,TXN0000005,2026-01-01 00:35:07,590.03,INDUSIND,PNB,UPI,PhonePe


In [16]:
df["bank_health"] = "NORMAL"

incidents = [
    ("SBI", "2026-01-05 10:00", "2026-01-05 18:00"),
    ("HDFC", "2026-01-10 14:00", "2026-01-10 22:00"),
    ("AXIS", "2026-01-15 08:00", "2026-01-15 16:00"),
    ("ICICI", "2026-01-20 10:00", "2026-01-20 18:00"),
    ("KOTAK", "2026-01-25 12:00", "2026-01-25 20:00"),
]

for bank, start, end in incidents:

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    duration = end - start

    degraded_start = start
    severe_start = start + duration * 0.25
    recovery_start = start + duration * 0.75

    # Degraded
    mask = (
        (df["receiver_bank"] == bank) &
        (df["timestamp"] >= degraded_start) &
        (df["timestamp"] < severe_start)
    )

    df.loc[mask, "bank_health"] = "DEGRADED"

    # Severe
    mask = (
        (df["receiver_bank"] == bank) &
        (df["timestamp"] >= severe_start) &
        (df["timestamp"] < recovery_start)
    )

    df.loc[mask, "bank_health"] = "SEVERE"

    # Recovery
    mask = (
        (df["receiver_bank"] == bank) &
        (df["timestamp"] >= recovery_start) &
        (df["timestamp"] <= end)
    )

    df.loc[mask, "bank_health"] = "RECOVERY"

print(df["bank_health"].value_counts())

bank_health
NORMAL      99194
SEVERE        397
RECOVERY      240
DEGRADED      169
Name: count, dtype: int64


In [17]:
# -----------------------------
# Generate realistic latency
# -----------------------------

base_latency = np.random.lognormal(
    mean=np.log(1000),
    sigma=0.35,
    size=NUM_TRANSACTIONS
)

df["latency_ms"] = base_latency

# Normal noise
normal_mask = df["bank_health"] == "NORMAL"

df.loc[normal_mask, "latency_ms"] *= np.random.uniform(
    0.8,
    1.3,
    normal_mask.sum()
)

# Degraded
degraded_mask = df["bank_health"] == "DEGRADED"

df.loc[degraded_mask, "latency_ms"] *= np.random.uniform(
    1.5,
    3.0,
    degraded_mask.sum()
)

# Severe
severe_mask = df["bank_health"] == "SEVERE"

df.loc[severe_mask, "latency_ms"] *= np.random.uniform(
    2.5,
    5.0,
    severe_mask.sum()
)

# Recovery
recovery_mask = df["bank_health"] == "RECOVERY"

df.loc[recovery_mask, "latency_ms"] *= np.random.uniform(
    1.2,
    2.0,
    recovery_mask.sum()
)

df["latency_ms"] = df["latency_ms"].round(2)

In [18]:
# -----------------------------
# Payment success probability
# -----------------------------

success_probability = np.full(
    NUM_TRANSACTIONS,
    0.975
)

# Small amount of natural variation
success_probability += np.random.normal(
    0,
    0.01,
    NUM_TRANSACTIONS
)

# Degraded
success_probability[
    degraded_mask
] = np.random.uniform(
    0.75,
    0.90,
    degraded_mask.sum()
)

# Severe
success_probability[
    severe_mask
] = np.random.uniform(
    0.40,
    0.65,
    severe_mask.sum()
)

# Recovery
success_probability[
    recovery_mask
] = np.random.uniform(
    0.85,
    0.95,
    recovery_mask.sum()
)

# Keep probabilities between 0 and 1
success_probability = np.clip(
    success_probability,
    0.30,
    0.995
)

random_values = np.random.random(
    NUM_TRANSACTIONS
)

df["payment_status"] = np.where(
    random_values < success_probability,
    "SUCCESS",
    "FAILED"
)

In [19]:
# -----------------------------
# Timeout generation
# -----------------------------

timeout_probability = np.where(
    df["latency_ms"] > 5000,
    0.80,
    np.where(
        df["latency_ms"] > 3000,
        0.35,
        0.02
    )
)

df["timeout"] = (
    np.random.random(NUM_TRANSACTIONS)
    < timeout_probability
).astype(int)

In [20]:
df["error_code"] = "NONE"

failed_mask = df["payment_status"] == "FAILED"

df.loc[failed_mask, "error_code"] = np.random.choice(
    [
        "BANK_TIMEOUT",
        "BANK_SERVER_ERROR",
        "NETWORK_TIMEOUT",
        "MERCHANT_ERROR",
        "INSUFFICIENT_FUNDS",
        "INVALID_UPI"
    ],
    failed_mask.sum(),
    p=[
        0.25,
        0.25,
        0.20,
        0.10,
        0.12,
        0.08
    ]
)

In [22]:
print("Dataset shape:")
print(df.shape)

print("\nPayment status:")
print(df["payment_status"].value_counts(normalize=True) * 100)

print("\nBank health:")
print(df["bank_health"].value_counts())

print("\nAverage latency:")
print(df["latency_ms"].mean())

print("\nTimeout rate:")
print(df["timeout"].mean() * 100)

Dataset shape:
(100000, 12)

Payment status:
payment_status
SUCCESS    97.243
FAILED      2.757
Name: proportion, dtype: float64

Bank health:
bank_health
NORMAL      99194
SEVERE        397
RECOVERY      240
DEGRADED      169
Name: count, dtype: int64

Average latency:
1132.5454112

Timeout rate:
2.161


In [23]:
df.to_csv(
    "../data/raw/transactions.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!
